In [2]:
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import numpy as np
import cv2

In [3]:
model_path = 'face_landmarker.task'

In [4]:
BaseOptions = mp.tasks.BaseOptions
FaceLandmarker = mp.tasks.vision.FaceLandmarker
FaceLandmarkerOptions = mp.tasks.vision.FaceLandmarkerOptions
VisionRunningMode = mp.tasks.vision.RunningMode

# Create a face landmarker instance with the video mode:
options = FaceLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=model_path),
    running_mode=VisionRunningMode.VIDEO)

In [5]:
def video_to_numpy(video_path, target_size=(224, 224), convert_rgb=True):
    """
    Reads a video clip from the DAiSEE dataset and returns a 4D NumPy array.
    Shape: (Num_Frames, Height, Width, Channels)
    """
    # Open the video file
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise IOError(f"Cannot open video file: {video_path}")
        
    frames = []
    
    while True:
        ret, frame = cap.read()
        if not ret:
            break  # Break the loop if the video ends or cannot be read
            
        # Optional: Resize the frame to reduce memory usage
        if target_size:
            frame = cv2.resize(frame, target_size)
            
        # OpenCV reads frames in BGR format by default; convert to RGB
        if convert_rgb:
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            
        frames.append(frame)
        
    cap.release()
    
    # Stack individual frames into a single 4D NumPy array
    video_array = np.stack(frames, axis=0)
    return video_array

# Example usage:
video_file = "DAiSEE/DataSet/Train/110001/1100011002/1100011002.avi"
video_np = video_to_numpy(video_file, target_size=(224, 224))

print("NumPy Array Shape:", video_np.shape)  # Output example: (300, 224, 224, 3)
print("Data Type:", video_np.dtype)  

NumPy Array Shape: (300, 224, 224, 3)
Data Type: uint8


## Train / validation / test face-landmark datasets

The cells below are self-contained: they build a manifest for each of
DAiSEE's three official splits (`Train`, `Validation`, `Test`), run every
clip through the face landmarker to get a fixed-length landmark sequence,
cache each clip's sequence as a `.npy` file, and wrap the three caches in
PyTorch `Dataset`/`DataLoader` objects. They reuse `FaceLandmarker`,
`options`, `mp`, `cv2`, and `np` from the cells above, and don't depend on
the scratch cells that crashed the kernel earlier.

In [6]:
from pathlib import Path

import pandas as pd
from tqdm.auto import tqdm

import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
# --- Paths & config -----------------------------------------------------
DAISEE_ROOT = Path("DAiSEE")
LABELS_DIR = DAISEE_ROOT / "Labels"
DATASET_DIR = DAISEE_ROOT / "DataSet"

# split key -> (DAiSEE's folder name, labels CSV)
SPLITS = {
    "train": ("Train", LABELS_DIR / "TrainLabels.csv"),
    "val": ("Validation", LABELS_DIR / "ValidationLabels.csv"),
    "test": ("Test", LABELS_DIR / "TestLabels.csv"),
}

LANDMARKS_DIR = Path("landmarks")  # ml/landmarks/{train,val,test}/<clip_id>.npy
LANDMARKS_DIR.mkdir(exist_ok=True)

LABEL_COLUMNS = ["Boredom", "Engagement", "Confusion", "Frustration"]

NUM_FRAMES = 30  # frames sampled per clip (uniform stride) -> fixed-length sequence


In [8]:
def build_video_index(dataset_split_dir: Path) -> dict[str, Path]:
    """Map ClipID (e.g. '1100011002.avi') -> full path to the .avi file.

    Walking the tree once and indexing by filename is more robust than
    reconstructing the path from the subject-id prefix, since a few subject
    folders in DAiSEE don't follow the 6-digit convention.
    """
    index = {}
    for avi_path in dataset_split_dir.rglob("*.avi"):
        index[avi_path.name] = avi_path
    return index


def load_split_manifest(split_key: str) -> pd.DataFrame:
    """Load a split's label CSV and attach the resolved video path to each row."""
    subdir_name, labels_csv = SPLITS[split_key]
    df = pd.read_csv(labels_csv)
    df.columns = df.columns.str.strip()  # "Frustration " has a trailing space in the source CSV

    video_index = build_video_index(DATASET_DIR / subdir_name)
    df["video_path"] = df["ClipID"].map(video_index)

    missing = df["video_path"].isna().sum()
    if missing:
        print(f"[{split_key}] warning: {missing} clip(s) listed in the labels CSV were not found on disk")
        df = df.dropna(subset=["video_path"]).reset_index(drop=True)

    return df


train_manifest = load_split_manifest("train")
val_manifest = load_split_manifest("val")
test_manifest = load_split_manifest("test")

for _name, _df in [("train", train_manifest), ("val", val_manifest), ("test", test_manifest)]:
    print(f"{_name}: {len(_df)} clips")


[train] warning: 506 clip(s) listed in the labels CSV were not found on disk
[test] warning: 146 clip(s) listed in the labels CSV were not found on disk
train: 4852 clips
val: 1429 clips
test: 1638 clips


In [9]:
def extract_landmarks(video_path: Path, num_frames: int = NUM_FRAMES) -> "np.ndarray | None":
    """Sample `num_frames` evenly-spaced frames from a clip and run the face
    landmarker on each. Returns an array of shape (num_frames, num_landmarks, 3)
    of (x, y, z) landmark coordinates, or None if no face was ever detected
    in the sampled frames.

    A fresh FaceLandmarker is created per video: VIDEO running mode requires
    strictly increasing timestamps on a single landmarker instance, and giving
    each clip its own timeline (starting at t=0) keeps that simple.
    """
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        cap.release()
        raise IOError(f"Cannot open video file: {video_path}")

    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if frame_count <= 0:
        cap.release()
        return None

    sample_indices = np.linspace(0, frame_count - 1, num=num_frames).round().astype(int)
    sample_set = set(sample_indices.tolist())

    frames_by_index = {}
    idx = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if idx in sample_set:
            frames_by_index[idx] = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        idx += 1
    cap.release()

    landmarks_by_index = {}
    with FaceLandmarker.create_from_options(options) as landmarker:
        for i, frame_idx in enumerate(sample_indices):
            rgb = frames_by_index.get(int(frame_idx))
            if rgb is None:
                continue
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
            timestamp_ms = int(i * (1000 / fps))
            result = landmarker.detect_for_video(mp_image, timestamp_ms)
            if result.face_landmarks:
                face = result.face_landmarks[0]
                landmarks_by_index[i] = np.array([[lm.x, lm.y, lm.z] for lm in face], dtype=np.float32)

    if not landmarks_by_index:
        return None

    n_points = len(next(iter(landmarks_by_index.values())))
    sequence = np.full((num_frames, n_points, 3), np.nan, dtype=np.float32)
    for i, lm in landmarks_by_index.items():
        sequence[i] = lm

    # Fill frames with no detection from the nearest earlier detected frame so
    # the cached sequence has no NaNs (a blink, head turn, or a moment fully
    # out of frame happens occasionally in these webcam clips).
    detected = ~np.isnan(sequence[:, 0, 0])
    if detected.any():
        last_good = np.where(detected)[0][0]
        for i in range(num_frames):
            if detected[i]:
                last_good = i
            else:
                sequence[i] = sequence[last_good]

    return sequence


In [10]:
def process_split(split_key: str, manifest: pd.DataFrame, limit: "int | None" = None) -> pd.DataFrame:
    """Extract + cache landmarks for every clip in a split.

    Resumable: a clip whose .npy already exists on disk is skipped, so a
    partially-run split can be re-run safely (e.g. after raising `limit`,
    or after fixing an error partway through the full split).
    """
    out_dir = LANDMARKS_DIR / split_key
    out_dir.mkdir(parents=True, exist_ok=True)

    rows = manifest.iloc[:limit] if limit else manifest
    records = []
    failed = []

    for row in tqdm(rows.itertuples(index=False), total=len(rows), desc=split_key):
        clip_id = Path(row.ClipID).stem
        npy_path = out_dir / f"{clip_id}.npy"

        if not npy_path.exists():
            try:
                sequence = extract_landmarks(Path(row.video_path))
            except Exception as exc:
                failed.append((row.ClipID, str(exc)))
                continue
            if sequence is None:
                failed.append((row.ClipID, "no face detected in any sampled frame"))
                continue
            np.save(npy_path, sequence)

        records.append({
            "clip_id": clip_id,
            "npy_path": str(npy_path),
            "Boredom": row.Boredom,
            "Engagement": row.Engagement,
            "Confusion": row.Confusion,
            "Frustration": row.Frustration,
        })

    if failed:
        print(f"[{split_key}] {len(failed)} clip(s) skipped (no face detected / read error)")

    split_df = pd.DataFrame.from_records(records)
    split_df.to_csv(LANDMARKS_DIR / f"{split_key}_manifest.csv", index=False)
    return split_df


Run the cell below to build the caches. `LIMIT` caps how many clips per
split get processed — start with a small number to sanity-check the
pipeline before committing to a full run: the full splits are ~5.4k / 1.4k
/ 1.8k clips for train/val/test, and every sampled frame goes through the
face landmarker, so processing everything will take a while. Re-running
with a larger (or `None`) `LIMIT` picks up where it left off, since already
-cached clips are skipped.

In [11]:
LIMIT = 20  # set to None to process every clip in each split

train_landmarks = process_split("train", train_manifest, limit=LIMIT)
val_landmarks = process_split("val", val_manifest, limit=LIMIT)
test_landmarks = process_split("test", test_manifest, limit=LIMIT)

len(train_landmarks), len(val_landmarks), len(test_landmarks)


test: 100%|██████████| 20/20 [00:00<00:00, 50994.58it/s]


(20, 20, 20)

In [12]:
class DaiseeLandmarksDataset(Dataset):
    """Loads cached face-landmark sequences for one DAiSEE split.

    Each item is (landmarks, labels):
      - landmarks: FloatTensor, shape (NUM_FRAMES, num_landmarks * 3) when
        flatten=True (the default), else (NUM_FRAMES, num_landmarks, 3).
      - labels: LongTensor of the 4 DAiSEE affect scores (each 0-3), in the
        order Boredom, Engagement, Confusion, Frustration -- unless `target`
        names a single column (e.g. "Engagement") to return instead.
    """

    def __init__(self, manifest: pd.DataFrame, target: "str | None" = None, flatten: bool = True):
        self.manifest = manifest.reset_index(drop=True)
        self.target = target
        self.flatten = flatten

    def __len__(self) -> int:
        return len(self.manifest)

    def __getitem__(self, idx: int):
        row = self.manifest.iloc[idx]
        sequence = np.load(row["npy_path"])  # (NUM_FRAMES, num_landmarks, 3)
        if self.flatten:
            sequence = sequence.reshape(sequence.shape[0], -1)
        landmarks = torch.from_numpy(sequence).float()

        if self.target:
            labels = torch.tensor(row[self.target], dtype=torch.long)
        else:
            labels = torch.tensor(row[LABEL_COLUMNS].to_numpy(dtype="int64"), dtype=torch.long)

        return landmarks, labels


In [13]:
train_dataset = DaiseeLandmarksDataset(train_landmarks)
val_dataset = DaiseeLandmarksDataset(val_landmarks)
test_dataset = DaiseeLandmarksDataset(test_landmarks)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

landmarks_batch, labels_batch = next(iter(train_loader))
print("landmarks batch:", landmarks_batch.shape)  # (B, NUM_FRAMES, num_landmarks * 3)
print("labels batch:", labels_batch.shape)          # (B, 4)


landmarks batch: torch.Size([16, 30, 1434])
labels batch: torch.Size([16, 4])


In [35]:
import math

# Helper: Positional Encoding Module
class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0)) # Shape: (1, max_len, d_model)

    def forward(self, x):
        # Adds positional encodings up to seq_len (dimension 1)
        return x + self.pe[:, :x.size(1), :]

# Main Transformer Model
class TransformerWithPredictionHead(nn.Module):
    def __init__(self, input_dim, d_model, nhead, num_encoder_layers, num_decoder_layers, dim_feedforward, dropout, num_classes):
        super().__init__()
        
        # 1. Define Linear Projection (Fixes your AttributeError)
        self.input_projection = nn.Linear(input_dim, d_model)
        
        # 2. Positional Encoding
        self.pos_encoder = PositionalEncoding(d_model)
        
        # 3. Core Transformer
        self.transformer = nn.Transformer(
            d_model=d_model,
            nhead=nhead,
            num_encoder_layers=num_encoder_layers,
            num_decoder_layers=num_decoder_layers,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True
        )
        
        # 4. Prediction Head
        self.fc_out = nn.Linear(d_model, num_classes)

    def forward(self, src, tgt, src_mask=None, tgt_mask=None, memory_mask=None,
                src_key_padding_mask=None, tgt_key_padding_mask=None, memory_key_padding_mask=None):
        
        # Automatically fix (batch, features, seq_len) -> (batch, seq_len, features)
        if src.dim() == 3 and src.shape[1] == 1434 and src.shape[2] == 30:
            src = src.transpose(1, 2)
        if tgt.dim() == 3 and tgt.shape[1] == 1434 and tgt.shape[2] == 30:
            tgt = tgt.transpose(1, 2)

        # 1. Project continuous features (1434) to d_model (512)
        src = self.input_projection(src)
        tgt = self.input_projection(tgt)
        
        # 2. Add Positional Encoding
        src = self.pos_encoder(src)
        tgt = self.pos_encoder(tgt)
        
        # 3. Transformer Forward Pass
        transformer_out = self.transformer(
            src=src,
            tgt=tgt,
            src_mask=src_mask,
            tgt_mask=tgt_mask,
            memory_mask=memory_mask,
            src_key_padding_mask=src_key_padding_mask,
            tgt_key_padding_mask=tgt_key_padding_mask,
            memory_key_padding_mask=memory_key_padding_mask
        )
        
        # 4. Pooling & Classification Head
        pooled_out = transformer_out[:, -1, :]
        logits = self.fc_out(pooled_out)
        return logits

# Hyperparameters
input_dim = 1434       # Feature dimension of continuous inputs
d_model = 512          # Transformer hidden size
nhead = 8
num_encoder_layers = 6
num_decoder_layers = 6
dim_feedforward = 2048
dropout = 0.1
num_classes = 4

# Instantiate Model
transformer_model = TransformerWithPredictionHead(
    input_dim=input_dim,
    d_model=d_model,
    nhead=nhead,
    num_encoder_layers=num_encoder_layers,
    num_decoder_layers=num_decoder_layers,
    dim_feedforward=dim_feedforward,
    dropout=dropout,
    num_classes=num_classes
)

In [36]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

epochs = 20
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(transformer_model.parameters(), lr=0.001)

In [37]:
print("starting training...")
transformer_model.to(device)
for epoch in range(epochs):
    running_loss = 0
    for inputs, targets in train_loader:
        inputs, targets = inputs.float().to(device), targets.float().to(device)

        # 3. Clear gradients from the previous step
        optimizer.zero_grad(set_to_none=True)

        # 4. Forward pass: compute predicted outputs
        outputs = transformer_model(inputs, inputs)
        
        # 5. Compute loss
        loss = criterion(outputs, targets)
        
        # 6. Backward pass: compute gradients
        loss.backward()
        
        # 7. Update model weights
        optimizer.step()
        
        running_loss += loss.item()
    print(running_loss / len(train_loader))

starting training...
2.7488173886958975
2.3167519569396973
3.0156041085720062
0.7950875647366047
0.6565655097365379
0.5973687320947647
0.9691779017448425
0.622295618057251
0.6505811363458633
0.8935043215751648
1.0761613845825195
1.5371726006269455
1.1096215546131134
1.173867642879486
0.9805547595024109
0.6273946762084961
1.1117851734161377
0.6500249020755291
0.6311493441462517
0.9642431735992432


In [38]:
transformer_model.eval()
running_loss = 0
for inputs, targets in test_loader:
    inputs, targets = inputs.float().to(device), targets.float().to(device)
    # 4. Forward pass: compute predicted outputs
    outputs = transformer_model(inputs, inputs)
    
    # 5. Compute loss
    loss = criterion(outputs, targets)
    running_loss += loss.item()
print(running_loss / len(train_loader))

10.030309677124023
